# Day 19: Problem Decomposition

**Today's question:** how do you start a program that is too big to hold in your head at once?

Answer: you don't start with code. You break the problem into small steps, give each step a function with
**one job**, and **check** each function before you move on.

Run each cell with **Shift+Enter**. Before running a cell marked 🤔, predict what it will show.

## 1. The problem: a text-statistics tool

> Given a paragraph, show how many words and sentences it has, the average word length, the longest word and the
> most common word. Ignore capital letters and punctuation when looking at words.

In [ ]:
sample_text = (
    "Mei and Omar study Python every evening. Mei writes the code and Omar checks it! "
    "Do they ever stop? Only for tea, and then they study Python again."
)
print(sample_text)

Where would you even start? Writing 30 lines top to bottom is how the Quiz Game got long and tangled.

## 2. Top-down design: from the goal to steps

Start from the **goal** and write the steps in plain words (**pseudocode**). Any step that still feels big gets
broken down again.

```
print_report(text):
    get the words of text (lowercase, no punctuation)
    show the number of words
    show the number of sentences
    show the average word length
    show the longest word
    show the most common word

get the words of text:
    for each piece of text between spaces:
        clean it (lowercase, letters only)
        keep it if anything is left
```

Each step becomes a function with **one job**:

| Function | Gets | Gives back | Job |
|----------|------|------------|-----|
| `clean_word(word)` | `"Hello,"` | `"hello"` | lowercase, letters only |
| `get_words(text)` | a paragraph | list of clean words | split + clean |
| `count_sentences(text)` | a paragraph | a number | count `.` `!` `?` |
| `average_word_length(words)` | list of words | a number | average letters per word |
| `longest_word(words)` | list of words | a word | the longest one |
| `most_common_word(words)` | list of words | a word | the most frequent one |
| `print_report(text)` | a paragraph | nothing (prints) | the **only** function that prints |

## 3. Stubs: name and docstring first

Write the `def` line and docstring before the body. A function whose body is only a docstring is **valid
Python**: it runs and returns `None`. This is called a **stub**.

In [ ]:
def clean_word(word):
    """Return word in lowercase with every non-letter removed."""


print(clean_word("Hello,"))

## 4. Checking a function: `print(f(x) == expected)`

Before writing the body, decide what the right answers are and write them as **checks**. Each check prints
`True` or `False`.

In [ ]:
print(clean_word("Hello,") == "hello")
print(clean_word("WORLD!") == "world")
print(clean_word("don't") == "dont")
print(clean_word("42") == "")

All `False`: the stub does nothing yet. That is the point: now we have a clear target.

To keep only letters we need one new string method, the sibling of `.isdigit()` (Day 13):
**`.isalpha()`** is `True` when every character is a letter.

In [ ]:
print("a".isalpha(), "Z".isalpha(), "7".isalpha(), ",".isalpha(), "".isalpha())

In [ ]:
def clean_word(word):
    """Return word in lowercase with every non-letter removed."""
    cleaned = ""
    for char in word.lower():
        if char.isalpha():
            cleaned += char
    return cleaned

🤔 **Predict:** how many `True`s now?

In [ ]:
print(clean_word("Hello,") == "hello")
print(clean_word("WORLD!") == "world")
print(clean_word("don't") == "dont")
print(clean_word("42") == "")

Good checks cover a **normal case** and the **edge cases**: capitals, punctuation inside the word, and a word
with nothing left. One check is not enough.

## 5. Build the rest one function at a time

`get_words` **uses** `clean_word` (composition, Day 18). Because `clean_word` is already checked, we can trust it.

In [ ]:
def get_words(text):
    """Return a list of the cleaned words in text, skipping empty ones."""
    words = []
    for piece in text.split():
        word = clean_word(piece)
        if word != "":
            words.append(word)
    return words


print(get_words("Hi, Asha! Hi.") == ["hi", "asha", "hi"])
print(get_words("") == [])
print(get_words("42 - 7") == [])

In [ ]:
def count_sentences(text):
    """Return the number of sentences: the count of '.', '!' and '?' characters."""
    return text.count(".") + text.count("!") + text.count("?")


print(count_sentences("Hi. How are you? Great!") == 3)
print(count_sentences("no punctuation") == 0)

🤔 What would `count_sentences("Wait... what?!")` give? (Try it.) Is that a bug or an acceptable limitation for
our tool? Either fix it or **write the limitation in the docstring**. Checks make you decide.

Now a function with a hidden problem:

In [ ]:
def average_word_length(words):
    """Return the average number of letters per word."""
    total = 0
    for word in words:
        total += len(word)
    return total / len(words)


print(average_word_length(["hi", "asha"]) == 3.0)

The normal case passes. 🤔 **Predict** the edge case: a text with no words at all.

In [ ]:
print(average_word_length([]) == 0)

The edge-case check found a crash before a user did. Fix it, then re-run **all** the checks for this function:

In [ ]:
def average_word_length(words):
    """Return the average number of letters per word (0 if there are no words)."""
    if len(words) == 0:
        return 0
    total = 0
    for word in words:
        total += len(word)
    return total / len(words)


print(average_word_length(["hi", "asha"]) == 3.0)
print(average_word_length([]) == 0)

In [ ]:
def longest_word(words):
    """Return the longest word (the first one if there is a tie, "" if there are no words)."""
    longest = ""
    for word in words:
        if len(word) > len(longest):
            longest = word
    return longest


print(longest_word(["tea", "python", "code"]) == "python")
print(longest_word(["cat", "dog"]) == "cat")
print(longest_word([]) == "")

In [ ]:
def most_common_word(words):
    """Return the word that appears most often (the first one found if there is a tie)."""
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    best_word = ""
    best_count = 0
    for word, count in counts.items():
        if count > best_count:
            best_word = word
            best_count = count
    return best_word


print(most_common_word(["tea", "milk", "tea"]) == "tea")
print(most_common_word([]) == "")

## 6. Assemble: the report

Only this function prints. It reads exactly like the pseudocode from section 2.

In [ ]:
def print_report(text):
    """Print a short statistics report for text."""
    words = get_words(text)
    print(f"Words:            {len(words)}")
    print(f"Sentences:        {count_sentences(text)}")
    print(f"Avg word length:  {average_word_length(words):.1f}")
    print(f"Longest word:     {longest_word(words)}")
    print(f"Most common word: {most_common_word(words)}")


print_report(sample_text)

"and" is the most common word, which is true but boring. Improving that (ignoring small words) would be a change
to **one** function, and its checks would tell us we didn't break it.

## 7. Common mistakes

**Mistake 1: a function that prints instead of returning cannot be checked.**

In [ ]:
def shout(text):
    """Return text in capitals with an exclamation mark."""
    print(text.upper() + "!")


print(shout("hi") == "HI!")

It *shows* `HI!` but the check says `False`: the function returned `None`.

**Mistake 2: the right answer with the wrong type.**

In [ ]:
print(len(get_words("Mei likes tea")) == "3")

`3` and `"3"` are different values, so the check fails even though the function works. Compare with the type the
function really returns.

**Mistake 3: only checking the easy case.** `average_word_length` passed its normal check and still crashed on an
empty list (section 5). Always add at least one edge case: empty input, a tie, a boundary value.
## 8. Planning Hangman together

**Rules.** The computer picks a secret word. The player sees one `_` per letter and guesses one letter at a
time. A correct letter is revealed everywhere it appears. A wrong letter costs a life. 6 wrong guesses: game
over. All letters found: the player wins. Then: play again?

**Pseudocode (top-down):**

```
play_game():
    repeat until the player says no:
        choose a random secret word
        play one round with it
        ask whether to play again

play_round(secret_word):
    no letters guessed yet, 0 wrong guesses
    while wrong guesses < 6 and the word is not fully guessed:
        show the word with blanks (and the lives left)
        ask for a valid guess and remember it
        if the guess is in the word: say so
        otherwise: one more wrong guess
    say win or lose and reveal the word
```

**Function plan:**

| Function | Gets | Gives back | Job | Check with `print(... == ...)`? |
|----------|------|------------|-----|------|
| `choose_word(words)` | list of words | one word | pick the secret word | partly (it's random) |
| `display_progress(secret_word, guessed_letters)` | word, list | `"p _ _ _ o _"` | build the hidden view | yes |
| `is_word_guessed(secret_word, guessed_letters)` | word, list | `True`/`False` | has the player won? | yes |
| `is_valid_guess(guess, guessed_letters)` | text, list | `True`/`False` | is it one new letter? | yes |
| `get_guess(guessed_letters)` | list | a letter | ask until the guess is valid | no: uses `input()` |
| `play_round(secret_word)` | word | `True` if won | one whole game | no: uses `input()` |
| `ask_play_again()` | nothing | `True`/`False` | ask y/n until valid | no: uses `input()` |
| `play_game()` | nothing | nothing | the whole program | no |

**Design rule:** keep `input()` and `print()` in a few functions, and make the others **return** values. Then
most of the game can be checked without playing it. That's why `get_guess` *calls* `is_valid_guess` instead of
doing all the work itself.

Stubs and checks are written **before** the code. They print `False` today; tomorrow you make them `True`.

In [ ]:
def display_progress(secret_word, guessed_letters):
    """Return the secret word with unguessed letters shown as _, e.g. 'p _ _ _ o _'."""


print(display_progress("python", ["p", "o"]) == "p _ _ _ o _")
print(display_progress("cat", []) == "_ _ _")

## Recap

1. **Top-down:** goal → steps in plain words (pseudocode) → one function per step.
2. **One job per function.** Functions that calculate return; only one or two functions talk to the user.
3. **Stub first** (`def` + docstring), then **checks** (`print(f(x) == expected)`), then the body.
4. A `False` check means the function **or** your expectation is wrong. Find out which.
5. `.isalpha()` is `True` when every character is a letter.

**Next time:** Project 02, Hangman, built function by function from today's plan.